# Treino do Condor

Ensina o **seu jeito** ao modelo local do Condor (Qwen3 4B), usando as conversas que você avaliou no chat.

1. Menu **Ambiente de execução → Alterar tipo → GPU T4** (grátis).
2. Rode as células em ordem (▶). A 2ª pede o arquivo `condor-treino-*.jsonl` que o Condor exportou (aba Sistema → Exportar para treino).
3. No fim baixa um `.gguf`. No PC: `.venv\Scripts\python.exe scripts\instalar_modelo_treinado.py arquivo.gguf` e escolha **CONDOR TREINADO** na aba Sistema.

Leva ~15–40 min conforme a quantidade de exemplos. Os dados ficam só nesta sessão do Colab; ela é apagada quando você fecha.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from google.colab import files
enviado = files.upload()
ARQUIVO = next(iter(enviado))
print("Dataset:", ARQUIVO)

## Modelo base
Mesmo modelo que o Condor usa hoje, carregado em 4 bits (LoRA: só uma camada pequena é treinada).

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 4096  # o prompt do Condor tem ~2.100 tokens + a conversa
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=ARQUIVO, split="train")

def formatar(lote):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in lote["messages"]]}

ds = ds.map(formatar, batched=True, remove_columns=ds.column_names)
print(len(ds), "exemplos")
print(ds[0]["text"][-1200:])

## Treino
Poucos exemplos → mais épocas. Só as respostas do Condor contam para o aprendizado (não o prompt).

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

EPOCAS = 3 if len(ds) < 300 else 2
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=EPOCAS,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

## Teste rápido

In [ ]:
FastLanguageModel.for_inference(model)
teste = [{"role": "user", "content": "Condor, quem é você e quem é o seu dono?"}]
entrada = tokenizer.apply_chat_template(teste, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
saida = model.generate(input_ids=entrada, max_new_tokens=160, temperature=0.7, top_p=0.8)
print(tokenizer.decode(saida[0][entrada.shape[1]:], skip_special_tokens=True))

## Exportar para o Ollama
Q4_K_M: o mesmo tamanho e velocidade do modelo que você já usa (~2,5 GB).

In [ ]:
import glob
model.save_pretrained_gguf("condor_gguf", tokenizer, quantization_method="q4_k_m")
gguf = sorted(glob.glob("condor_gguf/**/*.gguf", recursive=True) + glob.glob("*.gguf"))
gguf = [g for g in gguf if "q4_k_m" in g.lower()] or gguf
print(gguf)
files.download(gguf[0])